# UniMath — Python quickstart

`unimath` is a Cython extension over the UniMath C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install lituus-unimath
```

CI installs the wheel the release actually publishes and executes this
notebook against it, so a change that breaks the API breaks the build — but
only cell *execution* is checked, not that a printed value still matches
what's committed here.

## The API

In [1]:
import unimath

unimath.version(), unimath.__version__

('1.1.0', '1.1.0')

## BigInt

Arbitrary-precision integers. Constructs from a Python `int` of any size;
arithmetic and comparison operators coerce a plain `int` operand.

In [2]:
from unimath import BigInt

a = BigInt(-123456789)
b = BigInt(1_000_000_000_000) * BigInt(1_000_000_000_000)
print("a =", a)
print("b =", b)
print("a * b =", a * b)
print("b // 7 =", b // BigInt(7))
print("b % 7 =", b % BigInt(7))

a = -123456789
b = 1000000000000000000000000
a * b = -123456789000000000000000000000000
b // 7 = 142857142857142857142857
b % 7 = 1


## Fixed

Q-format fixed-point: `Fixed(value, frac_bits=N)` stores `value` scaled by
`2^N`. `value()` converts back to a Python `float`.

In [3]:
from unimath import Fixed

x = Fixed(3, frac_bits=32)
y = Fixed(2, frac_bits=32)
print("x + y =", (x + y).value())
print("x * y =", (x * y).value())
print("x // y =", (x // y).value())

x + y = 5.0
x * y = 6.0
x // y = 1.5


## BigFloat

Arbitrary-precision binary floating point, constructed from a Python `float`.
`to_f64()` rounds back to the nearest `float`.

In [4]:
from unimath import BigFloat

fa = BigFloat(10.0)
fb = BigFloat(3.0)
print("fa + fb =", (fa + fb).to_f64())
print("fa * fb =", (fa * fb).to_f64())
print("fa / fb =", (fa / fb).to_f64())

fa + fb = 13.0
fa * fb = 30.0
fa / fb = 3.3333333333333335


## Rational

Exact fractions. `Rational(num, den)` always reduces to lowest terms with a
positive denominator; `numerator()`/`denominator()` return the reduced
`BigInt` pair.

In [5]:
from unimath import Rational

ra = Rational(1, 2)
rb = Rational(1, 3)
print("1/2 + 1/3 =", (ra + rb).to_f64())
print("1/2 * 1/3 =", (ra * rb).to_f64())
red = Rational(4, 8)
print("4/8 reduces to", red.numerator(), "/", red.denominator())

1/2 + 1/3 = 0.8333333333333334
1/2 * 1/3 = 0.16666666666666666
4/8 reduces to 1 / 2


## Interval

Directed-rounding intervals: every operation widens its result so it
encloses the true value, even under float rounding.

In [6]:
from unimath import Interval

ia = Interval(1.0, 2.0)
ib = Interval(3.0, 4.0)
print("ia + ib =", ia + ib)
print("sqrt([4, 9]) =", Interval(4.0, 9.0).sqrt())

ia + ib = [3.9999999999999996, 6.000000000000001]
sqrt([4, 9]) = [1.9999999999999998, 3.0000000000000004]


## Complex

The square root of a negative number is not real, and neither is the
logarithm. `unimath.sqrt` and `unimath.log` return a complex there instead of
raising, and the argument's type decides which one: a `float` gives a `float`
or a builtin `complex`, a `BigFloat` gives a `BigFloat` or a `BigComplex`, and
so on down the backends.

The Nim core cannot do this — it resolves return types at compile time, so it
exposes a separately named `csqrt`. Python decides per value, so the choice
lives here.

In [7]:
from unimath import sqrt, log

print("sqrt(-1) =", sqrt(-1))
print("sqrt(4)  =", sqrt(4))
print("log(-1)  =", log(-1))

sqrt(-1) = 1j
sqrt(4)  = 2.0
log(-1)  = 3.141592653589793j


Arithmetic over `float64` takes and returns Python's builtin `complex`, so
results feed straight into `cmath` or NumPy. Division uses Smith's algorithm:
the textbook `(ac+bd)/(c²+d²)` overflows to NaN long before the quotient
itself does.

In [8]:
from unimath import ComplexMath

cm = ComplexMath()
print("abs(3+4i)   =", cm.abs(3 + 4j))
print("sqrt(-3-4i) =", cm.sqrt(-3 - 4j))
print("exp(i*pi)   =", cm.exp(complex(0, 3.141592653589793)))
huge = complex(1e300, 1e300)
print("huge/huge   =", cm.div(huge, huge))

abs(3+4i)   = 5.0
sqrt(-3-4i) = (1-2j)
exp(i*pi)   = (-1+1.2246467991473532e-16j)
huge/huge   = (1+0j)


The other backends have their own classes. `RationalComplex` keeps
`+ - * /`, `conj`, `norm2` and integer powers exact — a Gaussian rational
never leaves its field — while `BigComplex` carries the full transcendental
set at arbitrary precision.

In [9]:
from unimath import BigComplex, RationalComplex, Rational

z = RationalComplex(Rational(1, 2), Rational(3, 4))
print("norm2(1/2+3/4i) =", z.norm2())
print("(1/2+3/4i)^2    =", z ** 2)
print("abs(3+4i) big   =", float(BigComplex(3, 4).abs()))
print("ln(-1) big      =", complex(BigComplex(-1, 0).ln()))

norm2(1/2+3/4i) = 13/16
(1/2+3/4i)^2    = RationalComplex(Rational(-5, 16), Rational(3, 4))
abs(3+4i) big   = 5.0
ln(-1) big      = 3.141592653589793j
